In [1]:
import os
import re
import json
import gzip
import heapq
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from docx import Document
from tqdm.auto import tqdm

from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

JD_PATH = r"dataset/job_description.docx"
CANDIDATE_PATH = r"dataset/candidates.jsonl"
OUTPUT_PATH = "submission2.csv"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(MODEL_NAME)

TOP_K = 100

heap = []

print(embedder.get_sentence_embedding_dimension())

c:\Users\tahmi\Documents\Work\HACKATHONS\indiarun\pub\[PUB] India_runs_data_and_ai_challenge\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8284.05it/s]


384


In [2]:
def read_docx(path):
    doc = Document(path)
    return "\n".join(
        p.text.strip()
        for p in doc.paragraphs
        if p.text.strip()
    )

jd_text = read_docx(JD_PATH)

jd_text = re.sub(r"\s+", " ", jd_text).strip()

experience = None

m = re.search(r'(\d+)\s*[-–]\s*(\d+)\s*years', jd_text, re.I)

if m:
    experience = {
        "min": int(m.group(1)),
        "max": int(m.group(2)),
        "ideal": (int(m.group(1)) + int(m.group(2))) / 2
    }

sentences = re.split(r'(?<=[.!?])\s+', jd_text)

sentences = [
    s.strip()
    for s in sentences
    if len(s.strip()) > 15
]

mandatory = []
preferred = []
negative = []

mandatory_words = [
    "must",
    "required",
    "need",
    "strong",
    "hands-on",
    "production experience",
    "absolutely"
]

preferred_words = [
    "preferred",
    "nice to have",
    "good to have",
    "plus",
    "would like",
    "bonus"
]

negative_words = [
    "do not",
    "don't",
    "will not",
    "not want",
    "won't",
    "avoid",
    "disqualifier"
]

for s in sentences:

    ls = s.lower()

    if any(x in ls for x in mandatory_words):
        mandatory.append(s)

    elif any(x in ls for x in preferred_words):
        preferred.append(s)

    elif any(x in ls for x in negative_words):
        negative.append(s)

important_sentences = mandatory + preferred + negative

jd_embedding = embedder.encode(
    jd_text,
    normalize_embeddings=True
)

requirement_embeddings = embedder.encode(
    important_sentences,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=False
)

JD = {

    "text": jd_text,

    "experience": experience,

    "mandatory": mandatory,

    "preferred": preferred,

    "negative": negative,

    "requirements": important_sentences,

    "requirement_embeddings": requirement_embeddings,

    "embedding": jd_embedding

}

print(len(JD["requirements"]))
print(JD["experience"])

22
{'min': 5, 'max': 9, 'ideal': 7.0}


In [3]:
def build_candidate(candidate):

    profile = candidate.get("profile", {})

    career = candidate.get("career_history", [])

    skills = candidate.get("skills", [])

    education = candidate.get("education", [])

    career_text = " ".join(

        " ".join([

            job.get("title", ""),

            job.get("company", ""),

            job.get("industry", ""),

            job.get("description", "")

        ])

        for job in career

    )

    skill_text = " ".join(

        skill.get("name", "")

        for skill in skills

    )

    education_text = " ".join(

        " ".join([

            edu.get("degree", ""),

            edu.get("field_of_study", ""),

            edu.get("institution", "")

        ])

        for edu in education

    )

    document = " ".join([

        profile.get("headline", ""),

        profile.get("summary", ""),

        career_text,

        skill_text,

        education_text

    ])

    document = re.sub(r"\s+", " ", document).strip()

    embedding = embedder.encode(

        document,

        normalize_embeddings=True,

        show_progress_bar=False

    )

    return {

        "candidate_id": candidate["candidate_id"],

        "document": document,

        "embedding": embedding,

        "experience": profile.get("years_of_experience", 0),

        "skills": {

            s.get("name", "").lower()

            for s in skills

        },

        "career": career,

        "signals": candidate["redrob_signals"],

        "profile": profile

    }


def candidate_iterator(path):

    if path.endswith(".gz"):

        f = gzip.open(path, "rt", encoding="utf-8")

    else:

        f = open(path, "r", encoding="utf-8")

    with f:

        for line in f:

            yield build_candidate(

                json.loads(line)

            )

In [4]:
def cosine(a, b):
    return float(np.dot(a, b))


def experience_score(exp):

    if JD["experience"] is None:
        return 1.0

    mn = JD["experience"]["min"]
    mx = JD["experience"]["max"]
    ideal = JD["experience"]["ideal"]

    if exp < mn:
        return max(0.0, 1 - (mn - exp) / mn)

    if exp > mx:
        return max(0.0, 1 - (exp - mx) / mx)

    return 1 - abs(exp - ideal) / max(ideal - mn, mx - ideal, 1)


def semantic_score(candidate):

    global_sim = cosine(
        JD["embedding"],
        candidate["embedding"]
    )

    req_sim = np.dot(
        JD["requirement_embeddings"],
        candidate["embedding"]
    )

    return {

        "global": global_sim,

        "req_mean": float(req_sim.mean()),

        "req_max": float(req_sim.max()),

        "req_min": float(req_sim.min())
    }


def behavior_score(signals):

    score = 0.0

    score += signals["profile_completeness_score"] / 100

    score += signals["recruiter_response_rate"]

    score += signals["interview_completion_rate"]

    score += min(
        signals["github_activity_score"],
        100
    ) / 100 if signals["github_activity_score"] >= 0 else 0

    score += min(
        signals["search_appearance_30d"],
        200
    ) / 200

    score += min(
        signals["saved_by_recruiters_30d"],
        20
    ) / 20

    score += 1 if signals["open_to_work_flag"] else 0

    score += 1 if signals["verified_email"] else 0

    score += 1 if signals["verified_phone"] else 0

    score += 1 if signals["linkedin_connected"] else 0

    return score / 10


def penalty_score(candidate):

    s = candidate["signals"]

    penalty = 0

    if s["notice_period_days"] > 90:
        penalty += 0.10

    if s["avg_response_time_hours"] > 168:
        penalty += 0.10

    if s["recruiter_response_rate"] < 0.15:
        penalty += 0.10

    if candidate["experience"] < 1:
        penalty += 0.20

    return penalty


def skill_overlap(candidate):

    if len(candidate["skills"]) == 0:
        return 0

    jd_words = {

        w.lower()

        for w in re.findall(
            r"[A-Za-z][A-Za-z0-9+#.-]+",
            JD["text"]
        )

    }

    inter = len(
        jd_words.intersection(
            candidate["skills"]
        )
    )

    return inter / len(candidate["skills"])


def feature_vector(candidate):

    sem = semantic_score(candidate)

    return {

        "semantic_global": sem["global"],

        "semantic_mean": sem["req_mean"],

        "semantic_max": sem["req_max"],

        "semantic_min": sem["req_min"],

        "experience": experience_score(
            candidate["experience"]
        ),

        "behavior": behavior_score(
            candidate["signals"]
        ),

        "skill_overlap": skill_overlap(
            candidate
        ),

        "penalty": penalty_score(
            candidate
        )

    }

In [5]:
WEIGHTS = {

    "semantic_global": 0.25,

    "semantic_mean": 0.30,

    "semantic_max": 0.10,

    "semantic_min": 0.05,

    "experience": 0.10,

    "behavior": 0.15,

    "skill_overlap": 0.10,

    "penalty": 0.05

}


def final_score(candidate):

    f = feature_vector(candidate)

    score = (

        WEIGHTS["semantic_global"] * f["semantic_global"]

        + WEIGHTS["semantic_mean"] * f["semantic_mean"]

        + WEIGHTS["semantic_max"] * f["semantic_max"]

        + WEIGHTS["semantic_min"] * f["semantic_min"]

        + WEIGHTS["experience"] * f["experience"]

        + WEIGHTS["behavior"] * f["behavior"]

        + WEIGHTS["skill_overlap"] * f["skill_overlap"]

        - WEIGHTS["penalty"] * f["penalty"]

    )

    return score, f


def generate_reason(candidate, features):

    reasons = []

    if features["semantic_mean"] > 0.60:
        reasons.append("Strong semantic match with the job description")

    elif features["semantic_mean"] > 0.45:
        reasons.append("Good semantic alignment")

    if features["experience"] > 0.90:
        reasons.append(
            f'{candidate["experience"]:.1f} years of relevant experience'
        )

    if features["behavior"] > 0.75:
        reasons.append("Excellent recruiter engagement signals")

    elif features["behavior"] > 0.55:
        reasons.append("Good platform activity")

    if features["skill_overlap"] > 0.25:
        reasons.append("Relevant technical skill overlap")

    if len(reasons) == 0:
        reasons.append("Partial match with the role requirements")

    return ". ".join(reasons[:2]) + "."


def update_heap(candidate):

    score, features = final_score(candidate)

    row = (

        score,

        {

            "candidate_id": candidate["candidate_id"],

            "score": float(score),

            "reasoning": generate_reason(
                candidate,
                features
            )

        }

    )

    if len(heap) < TOP_K:

        heapq.heappush(heap, row)

    else:

        if score > heap[0][0]:

            heapq.heapreplace(heap, row)

In [6]:
rows = []

for candidate in tqdm(candidate_iterator(CANDIDATE_PATH)):

    update_heap(candidate)

rows = [x[1] for x in heap]

rows = sorted(
    rows,
    key=lambda x: (
        -x["score"],
        x["candidate_id"]
    )
)

submission = pd.DataFrame(rows)

submission["rank"] = np.arange(
    1,
    len(submission) + 1
)

submission = submission[
    [
        "candidate_id",
        "rank",
        "score",
        "reasoning"
    ]
]

submission["score"] = submission["score"].round(6)

submission.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print(submission.head())

print()

print("Total Selected :", len(submission))

print("Saved :", OUTPUT_PATH)

100000it [1:26:28, 19.27it/s]


   candidate_id  rank     score  \
0  CAND_0018499     1  0.545876   
1  CAND_0037566     2  0.541034   
2  CAND_0099806     3  0.539561   
3  CAND_0084819     4  0.536134   
4  CAND_0077337     5  0.535486   

                                           reasoning  
0  Excellent recruiter engagement signals. Releva...  
1  6.9 years of relevant experience. Excellent re...  
2  4.6 years of relevant experience. Excellent re...  
3            Excellent recruiter engagement signals.  
4  7.0 years of relevant experience. Excellent re...  

Total Selected : 100
Saved : submission2.csv
